In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test_notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.ml.build import build_dl

In [ ]:
cfg = OmegaConf.load("src/config/tune.yaml")

train_dl, valid_dl, split_filenames = build_dl(cfg)
print(f"train batches: {len(train_dl)}  |  valid batches: {len(valid_dl)}")

In [ ]:
batch = next(iter(train_dl))
image_1 = batch["image_1"]
image_2 = batch["image_2"]
mask = batch["mask"]
print(f"image_1: {tuple(image_1.shape)}  image_2: {tuple(image_2.shape)}  mask: {tuple(mask.shape)}")

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def denorm(image_chw: np.ndarray) -> np.ndarray:
    x = image_chw * STD + MEAN
    return np.clip(x, 0.0, 1.0).transpose(1, 2, 0)


n = image_1.shape[0]
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
if n == 1:
    axes = axes[None, :]

for i in range(n):
    axes[i, 0].imshow(denorm(image_1[i].numpy()))
    axes[i, 0].set_title("image_1")
    axes[i, 1].imshow(denorm(image_2[i].numpy()))
    axes[i, 1].set_title("image_2")
    axes[i, 2].imshow(mask[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("mask")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()